# Stage 2 — Category + Merchant Model (hybrid encoder / head / retrieval)

Trains the category head on the **Galahad v4 / Plaid PFCv1-aligned** dataset
produced by `05_taxonomy_v4_international.ipynb`. This is not PFCv2 and its
merchant-held-out scores are not a worldwide performance certification.
Architecture (per
`docs/transaction_model_strategy.md`, PRAGMA "borrow now" position):

1. **Encoder** — `all-MiniLM-L12-v2` contrastively fine-tuned (MNR loss) so
   transaction texts embed close to their `category | subcategory | merchant`
   label text. Input text is leakage-free key–value serialisation:
   `transaction: <desc> | type: <debit/credit> | payment: <pt> | amount_bucket: <b>`.
2. **Category head** — logistic-regression probe on frozen embeddings
   (PRAGMA-style embedding probe): fast, calibrated, easily retrained.
3. **Merchant retrieval** — per-merchant centroid index with cosine similarity
   and a validation-tuned threshold that says **"unknown merchant"** instead of
   forcing a false link (open-world behaviour for new banks).

**Generalisation stress tests** (used to expose cross-bank weaknesses):
- training rows are deduplicated by description *template*, so the model can't
  win by memorising reference numbers;
- splits are **template-held-out** (val/test) and **merchant-held-out**
  (12 merchants never seen in training) to measure transfer;
- 30% of contrastive pairs use description-only input so the model still works
  when a bank provides no payment-type/amount metadata;
- class-balanced sampling caps the mega-classes (bank fees, cash).

Deterministic (seed 42). Offline mode (models from local HF cache).
Smoke-test: set `GALAHAD_TRAIN_PFC_SMOKE=1`.

In [ ]:
from __future__ import annotations

import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import json
import logging
import random
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
HASH_KEY = "galahad-split-42"  # 16 chars, deterministic template splits

SMOKE = os.environ.get("GALAHAD_TRAIN_PFC_SMOKE", "0") == "1"
BASE_MODEL = "sentence-transformers/all-MiniLM-L12-v2"
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

CFG = {
    "max_batches": 2 if SMOKE else None,
    "max_variants_per_template": 3,
    "train_cap_per_category": 2_000 if SMOKE else 40_000,
    "mnr_pairs_cap": 3_000 if SMOKE else 200_000,
    "mnr_cap_per_category": 400 if SMOKE else 25_000,
    "desc_only_pair_fraction": 0.30,
    "lr_train_cap": 4_000 if SMOKE else 200_000,
    "eval_cap_per_split": 2_000 if SMOKE else 30_000,
    "epochs": 1,
    "batch_size": 64,
    "learning_rate": 2e-5,
    "n_holdout_merchants": 4 if SMOKE else 12,
    "merchant_exemplar_cap": 100,
    "merchant_min_train_rows": 25,
    "target_merchant_precision": 0.95,
}


def find_ml_model_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if cand.name == "ml_model" and (cand / "data" / "processed").is_dir():
            return cand
        if (cand / "ml_model" / "data" / "processed").is_dir():
            return cand / "ml_model"
    raise FileNotFoundError(f"ml_model root not found above {start}")


ML_MODEL_ROOT = find_ml_model_root(Path.cwd().resolve())
LABELS_PATH = ML_MODEL_ROOT / "data" / "processed" / "labels_v4" / "transactions_labelled_v4.parquet"
assert LABELS_PATH.is_file(), f"run notebooks 03+05 first: {LABELS_PATH} missing"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + ("_smoke" if SMOKE else "")
RUN_DIR = ML_MODEL_ROOT / "models" / f"pfc_v4_{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

log = logging.getLogger("stage2_train")
log.setLevel(logging.INFO)
log.handlers.clear()
_fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
for h in (logging.StreamHandler(), logging.FileHandler(RUN_DIR / "training.log", mode="w")):
    h.setFormatter(_fmt)
    log.addHandler(h)
log.info("device=%s smoke=%s run_dir=%s", DEVICE, SMOKE, RUN_DIR.name)

## 1. Collect template-deduplicated training rows (streaming)

One pass over `transactions_labelled_v3.parquet`: keep labelled rows
(`pfc_primary != UNCATEGORIZED`), take at most one raw variant per template per
batch, then cap at 3 variants per template globally. Template frequency is kept
as a feature-free popularity weight for analysis.

In [ ]:
USE_COLS = ["TransactionParticulars", "TransactionTypeDebitCredit", "PaymentType",
            "TransactionAmount", "pfc_primary", "pfc_detailed", "merchant_v3",
            "transaction_status", "template", "label_source", "label_confidence"]

pf = pq.ParquetFile(LABELS_PATH)
frames, tmpl_freq = [], Counter()
t0 = time.time()
for batch_idx, batch in enumerate(pf.iter_batches(batch_size=1_048_576, columns=USE_COLS)):
    if CFG["max_batches"] is not None and batch_idx >= CFG["max_batches"]:
        break
    df = batch.to_pandas()
    labelled = df[df["pfc_primary"] != "UNCATEGORIZED"]
    tmpl_freq.update(labelled["template"].value_counts().to_dict())
    frames.append(labelled.drop_duplicates(subset="template", keep="first"))
    if (batch_idx + 1) % 6 == 0:
        log.info("collect batch %d (%.0fs)", batch_idx + 1, time.time() - t0)

pool = pd.concat(frames, ignore_index=True)
pool = pool.groupby("template", sort=False).head(CFG["max_variants_per_template"]) \
           .reset_index(drop=True)
pool["template_freq"] = pool["template"].map(tmpl_freq)
del frames
log.info("pool: %s rows, %s templates (%.0fs)",
         f"{len(pool):,}", f"{pool['template'].nunique():,}", time.time() - t0)
pool["pfc_primary"].value_counts()

## 2. Model input text (leakage-free) and label text

In [ ]:
AMOUNT_BUCKETS = [(10, "lt_10"), (100, "10_100"), (1_000, "100_1k"),
                  (10_000, "1k_10k"), (100_000, "10k_100k"), (float("inf"), "gt_100k")]


def amount_bucket(a: pd.Series) -> pd.Series:
    out = pd.Series("unknown", index=a.index, dtype="object")
    prev = 0.0
    for hi, name in AMOUNT_BUCKETS:
        out[(a >= prev) & (a < hi)] = name
        prev = hi
    return out


def build_input_text(df: pd.DataFrame) -> pd.Series:
    desc = df["TransactionParticulars"].astype("string").str.lower().str.strip().fillna("")
    dc = df["TransactionTypeDebitCredit"].map({"d": "debit", "c": "credit"}).fillna("unknown")
    pt = df["PaymentType"].astype("string").fillna("unknown")
    ab = amount_bucket(df["TransactionAmount"].astype(float))
    return ("transaction: " + desc + " | type: " + dc
            + " | payment: " + pt + " | amount_bucket: " + ab)


def build_desc_only_text(df: pd.DataFrame) -> pd.Series:
    desc = df["TransactionParticulars"].astype("string").str.lower().str.strip().fillna("")
    return "transaction: " + desc


pool["input_text"] = build_input_text(pool)
pool["label_text"] = ("category: " + pool["pfc_primary"]
                      + " | subcategory: " + pool["pfc_detailed"].fillna("OTHER")
                      + " | merchant: " + pool["merchant_v3"].fillna("unknown"))
pool[["input_text", "label_text"]].head(4).to_dict(orient="records")

## 3. Splits — merchant-held-out and template-held-out

- 12 merchants (seeded pick, excluding the 3 highest-volume ones) are removed
  from training entirely → `test_unseen_merchant`.
- Remaining templates split 90/5/5 into train/val/test by deterministic hash.

In [ ]:
rng = np.random.default_rng(SEED)
merch_counts = pool["merchant_v3"].value_counts()
eligible = merch_counts.iloc[3:][merch_counts.iloc[3:] >= 200].index.to_numpy()
holdout_merchants = sorted(rng.choice(
    eligible, size=min(CFG["n_holdout_merchants"], len(eligible)), replace=False))
log.info("holdout merchants: %s", holdout_merchants)

is_holdout = pool["merchant_v3"].isin(holdout_merchants)
test_unseen = pool[is_holdout].copy()
rest = pool[~is_holdout].copy()

bucket = pd.util.hash_pandas_object(
    rest["template"].astype("string"), index=False, hash_key=HASH_KEY).to_numpy() % 100
rest["split"] = np.select([bucket < 90, bucket < 95], ["train", "val"], default="test")
train_df = rest[rest["split"] == "train"]
val_df = rest[rest["split"] == "val"]
test_df = rest[rest["split"] == "test"]

# class-balance the training pool (cap mega-classes)
train_df = (train_df.sample(frac=1.0, random_state=SEED)
            .groupby("pfc_primary", sort=False)
            .head(CFG["train_cap_per_category"]).reset_index(drop=True))

split_summary = pd.DataFrame({
    "train": train_df["pfc_primary"].value_counts(),
    "val": val_df["pfc_primary"].value_counts(),
    "test": test_df["pfc_primary"].value_counts(),
    "test_unseen_merchant": test_unseen["pfc_primary"].value_counts(),
}).fillna(0).astype(int)
log.info("splits: train=%s val=%s test=%s unseen=%s",
         f"{len(train_df):,}", f"{len(val_df):,}", f"{len(test_df):,}", f"{len(test_unseen):,}")
split_summary

## 4. Contrastive fine-tuning of the encoder (MNR loss)

Pairs `(input_text, label_text)` pull each transaction toward its
category/subcategory/merchant description; in-batch negatives push apart
different labels. 30% of pairs use description-only input for robustness to
metadata-poor banks.

In [ ]:
from datasets import Dataset
from sentence_transformers import (SentenceTransformer, SentenceTransformerTrainer,
                                   SentenceTransformerTrainingArguments, losses)

pairs = (train_df.sample(frac=1.0, random_state=SEED)
         .groupby("pfc_primary", sort=False).head(CFG["mnr_cap_per_category"]))
pairs = pairs.sample(n=min(CFG["mnr_pairs_cap"], len(pairs)), random_state=SEED)

desc_only_mask = rng.random(len(pairs)) < CFG["desc_only_pair_fraction"]
anchors = np.where(desc_only_mask,
                   build_desc_only_text(pairs).to_numpy(dtype=object),
                   pairs["input_text"].to_numpy(dtype=object))
train_ds = Dataset.from_dict({"anchor": list(anchors),
                              "positive": pairs["label_text"].tolist()})
log.info("MNR pairs: %s (%.0f%% desc-only)", f"{len(train_ds):,}",
         100 * desc_only_mask.mean())

model = SentenceTransformer(BASE_MODEL, device=DEVICE)
loss = losses.MultipleNegativesRankingLoss(model)
args = SentenceTransformerTrainingArguments(
    output_dir=str(RUN_DIR / "checkpoints"),
    num_train_epochs=CFG["epochs"],
    per_device_train_batch_size=CFG["batch_size"],
    learning_rate=CFG["learning_rate"],
    warmup_ratio=0.1,
    logging_steps=200,
    save_strategy="no",
    report_to=[],
    seed=SEED,
    dataloader_num_workers=0,
    disable_tqdm=True,
)
trainer = SentenceTransformerTrainer(model=model, args=args,
                                     train_dataset=train_ds, loss=loss)
t0 = time.time()
trainer.train()
log.info("MNR fine-tune done in %.0fs", time.time() - t0)
model.save(str(RUN_DIR / "encoder"))
log.info("encoder saved -> %s", RUN_DIR / "encoder")

## 5. Category head — logistic probe on frozen embeddings

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score


def encode(texts, name):
    t0 = time.time()
    emb = model.encode(list(texts), batch_size=256, show_progress_bar=False,
                       normalize_embeddings=True)
    log.info("encoded %s (%s) in %.0fs", f"{len(texts):,}", name, time.time() - t0)
    return np.asarray(emb, dtype=np.float32)


lr_train = (train_df.sample(frac=1.0, random_state=SEED)
            .head(CFG["lr_train_cap"]).reset_index(drop=True))
val_s = val_df.sample(n=min(CFG["eval_cap_per_split"], len(val_df)), random_state=SEED)
test_s = test_df.sample(n=min(CFG["eval_cap_per_split"], len(test_df)), random_state=SEED)
unseen_s = test_unseen.sample(n=min(CFG["eval_cap_per_split"], len(test_unseen)),
                              random_state=SEED)

X_train = encode(lr_train["input_text"], "train")
X_val = encode(val_s["input_text"], "val")
X_test = encode(test_s["input_text"], "test")
X_unseen = encode(unseen_s["input_text"], "unseen-merchant")

clf = LogisticRegression(max_iter=300, C=4.0, class_weight="balanced", n_jobs=-1)
t0 = time.time()
clf.fit(X_train, lr_train["pfc_primary"])
log.info("logistic head fit in %.0fs", time.time() - t0)

category_metrics = {}
for split_name, X, df_s in [("val", X_val, val_s), ("test", X_test, test_s),
                            ("test_unseen_merchant", X_unseen, unseen_s)]:
    y_true, y_pred = df_s["pfc_primary"], clf.predict(X)
    category_metrics[split_name] = {
        "n": int(len(y_true)),
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "macro_f1": round(f1_score(y_true, y_pred, average="macro"), 4),
        "weighted_f1": round(f1_score(y_true, y_pred, average="weighted"), 4),
    }
    log.info("category %s: %s", split_name, category_metrics[split_name])

(RUN_DIR / "category_report_test.txt").write_text(
    classification_report(test_s["pfc_primary"], clf.predict(X_test), zero_division=0))
pd.DataFrame(category_metrics).T

## 6. Confidence gate — auto-accept coverage vs precision

The app should auto-accept only confident predictions and route the rest to
fallback/review, so we report coverage at fixed precision targets.

In [ ]:
proba = clf.predict_proba(X_test)
conf = proba.max(axis=1)
pred = clf.classes_[proba.argmax(axis=1)]
correct = (pred == test_s["pfc_primary"].to_numpy())

gate_rows = []
for thr in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    mask = conf >= thr
    if mask.sum() == 0:
        continue
    gate_rows.append({"threshold": thr,
                      "coverage": round(float(mask.mean()), 4),
                      "precision": round(float(correct[mask].mean()), 4)})
gate_df = pd.DataFrame(gate_rows)
# floor at 0.5: even if precision passes at lower cutoffs, never auto-accept
# a prediction the model itself ranks below a coin flip
auto_accept_thr = max(0.5, next((r["threshold"] for r in gate_rows
                                 if r["precision"] >= 0.95), 0.95))
log.info("category auto-accept threshold (>=95%% precision on test, floor 0.5): %.2f",
         auto_accept_thr)

# behaviour on unseen merchants: opaque descriptions (account numbers, phone
# numbers) SHOULD land below the gate and route to review, not auto-accept
proba_u = clf.predict_proba(X_unseen)
conf_u = proba_u.max(axis=1)
pred_u = clf.classes_[proba_u.argmax(axis=1)]
correct_u = pred_u == unseen_s["pfc_primary"].to_numpy()
mask_u = conf_u >= auto_accept_thr
unseen_gate = {
    "auto_accept_threshold": float(auto_accept_thr),
    "unseen_auto_accept_coverage": round(float(mask_u.mean()), 4),
    "unseen_auto_accept_precision": round(float(correct_u[mask_u].mean()), 4)
        if mask_u.sum() else None,
    "unseen_routed_to_review": round(float(1 - mask_u.mean()), 4),
    "unseen_overall_accuracy_ungated": round(float(correct_u.mean()), 4),
}
log.info("unseen-merchant gate behaviour: %s", unseen_gate)
gate_df

## 7. Merchant retrieval index + unknown-merchant threshold

Per-merchant centroids from training embeddings; cosine top-k retrieval.

**Threshold tuning is leave-one-merchant-out (open-set protocol).** Each
validation row scores once normally ("known" score) and once with its own
merchant's centroid masked out ("unknown" score — simulating that merchant
being absent from the index), so all indexed merchants act as unknown-proxies.
The threshold maximises known coverage subject to ≥95% precision on accepted
knowns AND ≥80% leave-one-out unknown rejection. The untouched
merchant-held-out test split then measures true open-world rejection.

In [ ]:
merch_train = lr_train[lr_train["merchant_v3"].notna()].copy()
merch_counts_train = merch_train["merchant_v3"].value_counts()
kept_merchants = sorted(
    merch_counts_train[merch_counts_train >= CFG["merchant_min_train_rows"]].index)
merch_train = merch_train[merch_train["merchant_v3"].isin(kept_merchants)]


def build_index(merchants):
    cents, nms = [], []
    for m, grp in merch_train[merch_train["merchant_v3"].isin(merchants)] \
            .groupby("merchant_v3"):
        embs = X_train[grp.head(CFG["merchant_exemplar_cap"]).index.to_numpy()]
        c = embs.mean(axis=0)
        cents.append(c / np.linalg.norm(c))
        nms.append(m)
    return np.vstack(cents).astype(np.float32), nms


def retrieve(X, cents):
    sims = X @ cents.T
    order = np.argsort(-sims, axis=1)[:, :5]
    return order, np.take_along_axis(sims, order, axis=1)


# --- leave-one-merchant-out threshold tuning (standard open-set protocol):
# every val row with an indexed merchant contributes a "known" score (sim to
# its own centroid via normal retrieval) AND an "unknown" score (max sim with
# its own merchant's centroid masked out — simulating that merchant being
# absent from the index). This uses all merchants as unknown-proxies instead
# of a handful, so the threshold generalises to genuinely new merchants.
centroids, names = build_index(kept_merchants)
log.info("merchant index: %d merchants", len(names))
namearr_all = np.array(names, dtype=object)
name_to_col = {m: j for j, m in enumerate(names)}

val_r = val_s.reset_index(drop=True)
known_mask_v = val_r["merchant_v3"].isin(names).to_numpy()
Xv = X_val[known_mask_v]
val_merchants = val_r.loc[known_mask_v, "merchant_v3"].to_numpy()
sims_full = Xv @ centroids.T
own_cols = np.array([name_to_col[m] for m in val_merchants])
known_top = sims_full.max(axis=1)
correct_k = namearr_all[sims_full.argmax(axis=1)] == val_merchants
loo = sims_full.copy()
loo[np.arange(len(loo)), own_cols] = -1.0
unknown_sims = loo.max(axis=1)

# maximise known coverage subject to precision >= target on accepted knowns
# AND leave-one-out unknown-rejection >= the floor; fall back to max(accept+reject)
MIN_UNKNOWN_REJECT = 0.80
merchant_thr, best_cov, best_j, fallback_thr = 0.95, -1.0, -1.0, 0.95
for thr in np.arange(0.30, 0.96, 0.01):
    acc = known_top >= thr
    if acc.sum() == 0:
        break
    if correct_k[acc].mean() < CFG["target_merchant_precision"]:
        continue
    reject = float((unknown_sims < thr).mean())
    j = float(acc.mean()) + reject
    if j > best_j:
        best_j, fallback_thr = j, round(float(thr), 2)
    if reject >= MIN_UNKNOWN_REJECT and float(acc.mean()) > best_cov:
        best_cov, merchant_thr = float(acc.mean()), round(float(thr), 2)
if best_cov < 0:
    log.warning("no threshold met the %.0f%% unknown-reject floor; using max-J fallback",
                100 * MIN_UNKNOWN_REJECT)
    merchant_thr = fallback_thr
log.info("tuned merchant threshold: %.2f (LOO unknown-reject %.3f, known-accept %.3f)",
         merchant_thr, float((unknown_sims < merchant_thr).mean()),
         float((known_top >= merchant_thr).mean()))

test_m = test_s[test_s["merchant_v3"].isin(names)].reset_index(drop=True)
Xtm = X_test[test_s.reset_index(drop=True)["merchant_v3"].isin(names).to_numpy()]
order_t, sims_t = retrieve(Xtm, centroids)
namearr = np.array(names, dtype=object)
top1 = namearr[order_t[:, 0]]
in_top5 = (namearr[order_t] == test_m["merchant_v3"].to_numpy()[:, None]).any(axis=1)
acc_mask_t = sims_t[:, 0] >= merchant_thr

order_u, sims_u = retrieve(X_unseen, centroids)
merchant_metrics = {
    "n_index_merchants": len(names),
    "threshold": merchant_thr,
    "threshold_tuning": "leave-one-merchant-out on val",
    "loo_unknown_reject_val": round(float((unknown_sims < merchant_thr).mean()), 4),
    "seen_test_n": int(len(test_m)),
    "seen_top1_accuracy": round(float((top1 == test_m["merchant_v3"].to_numpy()).mean()), 4),
    "seen_top5_accuracy": round(float(in_top5.mean()), 4),
    "seen_accepted_coverage": round(float(acc_mask_t.mean()), 4),
    "seen_accepted_precision": round(
        float((top1 == test_m["merchant_v3"].to_numpy())[acc_mask_t].mean()), 4)
        if acc_mask_t.sum() else None,
    "unseen_n": int(len(X_unseen)),
    "unseen_correctly_rejected_rate": round(float((sims_u[:, 0] < merchant_thr).mean()), 4),
}
log.info("merchant metrics: %s", merchant_metrics)
merchant_metrics

## 8. Persist model artifacts & metrics

In [ ]:
joblib.dump(clf, RUN_DIR / "category_head.joblib")
np.savez_compressed(RUN_DIR / "merchant_index.npz",
                    centroids=centroids, names=np.array(names, dtype=object))

metrics = {
    "run_id": RUN_ID,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "smoke_test_mode": SMOKE,
    "base_model": BASE_MODEL,
    "device": DEVICE,
    "config": CFG,
    "data": {
        "source": str(LABELS_PATH),
        "pool_rows": int(len(pool)),
        "train_rows": int(len(train_df)),
        "mnr_pairs": int(len(train_ds)),
        "holdout_merchants": list(holdout_merchants),
    },
    "category": category_metrics,
    "category_confidence_gate": gate_rows,
    "category_auto_accept_threshold": auto_accept_thr,
    "category_unseen_merchant_gate": unseen_gate,
    "merchant": merchant_metrics,
}
(RUN_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2, default=str))
taxonomy_v4 = json.loads(
    (ML_MODEL_ROOT / "data" / "processed" / "labels_v4" / "taxonomy_v4.json").read_text())
(RUN_DIR / "label_classes.json").write_text(json.dumps(
    {"taxonomy": "pfc_v4", "categories": list(clf.classes_),
     "merchant_threshold": merchant_thr,
     "category_auto_accept_threshold": auto_accept_thr,
     # app-facing display names shipped with the artifact so the API stays
     # taxonomy-agnostic (primary map; detailed map included for later use)
     "app_category_map": taxonomy_v4["app_display"]["primary"],
     "app_detailed_map": taxonomy_v4["app_display"]["detailed"]}, indent=2))
log.info("artifacts saved under %s", RUN_DIR)
for f in sorted(RUN_DIR.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(RUN_DIR)!s:45s} {f.stat().st_size / 1024:10.1f} KB")

## 9. Sanity check — does it behave on unfamiliar bank formats?

A few hand-written strings in formats this dataset has *never seen* (South
African / generic international styles) to sanity-check cross-bank behaviour.

In [ ]:
DEMO = [
    ("POS PURCHASE CHECKERS SANDTON CITY 458212", "d", "ci", 350.0),
    ("NETFLIX.COM AMSTERDAM NL", "d", "ci", 199.0),
    ("SALARY PAYMENT - ACME PTY LTD", "c", "bt", 15000.0),
    ("ATM WITHDRAWAL FNB MAERUA MALL", "d", "ci", 500.0),
    ("KFC HYPER WINDHOEK", "d", "ci", 89.9),
    ("INSURANCE PREMIUM OLD MUTUAL POLICY 99231", "d", "bi", 450.0),
]
demo_df = pd.DataFrame(DEMO, columns=["TransactionParticulars",
                                      "TransactionTypeDebitCredit",
                                      "PaymentType", "TransactionAmount"])
demo_texts = build_input_text(demo_df)
Xd = encode(demo_texts, "demo")
proba_d = clf.predict_proba(Xd)
order_d, sims_d = retrieve(Xd, centroids)

for i, (desc, *_rest) in enumerate(DEMO):
    cat_i = proba_d[i].argmax()
    merch = names[order_d[i, 0]] if sims_d[i, 0] >= merchant_thr else "<unknown merchant>"
    print(f"{desc[:45]:47s} -> {clf.classes_[cat_i]:22s} "
          f"(p={proba_d[i, cat_i]:.2f})  merchant: {merch} (sim={sims_d[i, 0]:.2f})")

## 10. Notes & next steps

- **Metrics context**: category scores are on *template-held-out* data — the
  model never saw any variant of those description templates. The
  `test_unseen_merchant` split additionally removes the merchants entirely,
  approximating "a new bank's data".
- **Unseen-merchant category accuracy is structurally low, by design of the
  split**: most held-out-merchant rows are *opaque* descriptions (bare account
  numbers like `sbxxnictus02651923`, phone numbers for airtime) whose category
  is only knowable through merchant identity — knowledge the split removes on
  purpose. Semantic descriptions transfer well (supermarkets/clothing ~87–98%
  in diagnosis). The confidence gate is the intended handler: opaque rows
  should fall below the auto-accept threshold and route to review/rules — see
  `category_unseen_merchant_gate` in `metrics.json`.
- The old benchmark under `data/evaluation/` uses the v2 label space, so
  numbers are not directly comparable; rebuild it against taxonomy v3 with
  `evaluation/build_eval_set.py` as a follow-up.
- **Serving contract** (matches `docs/transaction_model_strategy.md`):
  rules → this model (category head + merchant retrieval + thresholds) →
  fallback. `label_classes.json` carries both thresholds.
- When account/customer identifiers arrive (app deployment), this encoder
  becomes the event encoder for the PRAGMA history stages deferred earlier.